# anticipated群：ARIS care 最初の該当記載日の再抽出
- 対象: unex_study.unex_death = 0
- 検索範囲: karte 全期間〜死亡日（死亡日当日を含む）
- Phase 1: ARIS全ヒット＋「今後」欄の文言変化を抽出 → 目視確認用CSV（2本）
- Phase 2: 目視確認後 → first_date 確定、24h/48h/72h/7日フラグ

In [ ]:
# 0. 設定
import sqlite3, re, unicodedata
import numpy as np
import pandas as pd
from pathlib import Path

DB  = "/Users/muna/Hana_research/data/db/Hana_Research.db"
OUT = Path("/Users/muna/Hana_research/data/output/aris_firstdate")
OUT.mkdir(parents=True, exist_ok=True)

# 死亡確認カルテの判定（Cell 2 の visit_type 一覧を見て修正）
DEATH_VISIT_TYPES = ["緊急", "往診", "休日", "定期", "深夜", "夜間往診"]   # 部分一致
DEATH_WORDS       = ["死亡確認", "死亡診断", "死亡を確認", "永眠"]        # 要確認・追加可
NURSE_TYPE        = "訪問看護"

THRESH_H  = {"24h": 24, "48h": 48, "72h": 72, "7d": 168}
YOTEI_WIN = 15   # ARIS表記の前後何文字以内の「予定」を yotei=1 とするか

# 「今後」欄（医師・定期カルテのみ）
TEIKI_TYPE = "定期"                              # visit_type 部分一致
KONGO_HEAD = r"今後(?:のこと|について)?"         # Cell 5b の表記ゆれ一覧を見て追加（例: r"今後(?:のこと|について|の方針)?"）
MARK = "◯○●#"                                   # 見出し記号（NFKC後。＃→#）

# NFKC正規化後に適用（全角ＡＲＩＳ・半角ｱﾘｽ も吸収）
ARIS_RE = re.compile(r"(?<![A-Za-z])A?RIS(?![A-Za-z])|アリス\s*・?\s*ケア", re.I)   # カタカナは「アリスケア」のみ（「アリス」単独は除外）

def nfkc(x):
    return unicodedata.normalize("NFKC", x) if isinstance(x, str) else ""

In [ ]:
# 1. 対象患者と死亡日
con = sqlite3.connect(DB)

coh = pd.read_sql("SELECT Patient_ID, Study_ID, unex_death FROM unex_study", con)
print("unex_study:", len(coh))
print(coh["unex_death"].value_counts(dropna=False))

ev = pd.read_sql("""
    SELECT DISTINCT Study_ID, death_date FROM event
    WHERE death_date IS NOT NULL AND death_date <> ''
""", con)
ev["death_date"] = pd.to_datetime(ev["death_date"], errors="coerce").dt.normalize()
ev = ev.dropna(subset=["death_date"]).drop_duplicates()
dup = ev[ev["Study_ID"].duplicated(keep=False)]
assert dup.empty, f"1患者に複数のdeath_date:\n{dup}"

ant = coh[coh["unex_death"] == 0].merge(ev, on="Study_ID", how="left")
print("anticipated:", len(ant), "/ death_date欠損:", ant["death_date"].isna().sum())

con.execute("DROP TABLE IF EXISTS temp.ids")
con.execute("CREATE TEMP TABLE ids (Study_ID TEXT PRIMARY KEY)")
con.executemany("INSERT INTO temp.ids VALUES (?)", [(s,) for s in ant["Study_ID"]])

In [ ]:
# 2. visit_type の確認（DEATH_VISIT_TYPES の修正用）
print(pd.read_sql("""
    SELECT k.visit_type, COUNT(*) AS n
    FROM karte k JOIN temp.ids i ON k.Study_ID = i.Study_ID
    GROUP BY k.visit_type ORDER BY n DESC
""", con).to_string())

In [ ]:
# 3. カルテ読込・日時パース・死亡日までに限定
kt = pd.read_sql("""
    SELECT k.record_no, k.Study_ID, k.visit_type, k.visit_datetime, k.karte_text
    FROM karte k JOIN temp.ids i ON k.Study_ID = i.Study_ID
""", con)

s = (kt["visit_datetime"].astype(str)
       .str.replace(r"\([^)]*\)", " ", regex=True)
       .str.replace(r"\s+", " ", regex=True).str.strip())
kt["rec_dt"]   = pd.to_datetime(s, format="%Y/%m/%d %H:%M", errors="coerce")
kt["rec_date"] = pd.to_datetime(s.str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})")[0],
                                format="%Y/%m/%d", errors="coerce")
print("日付パース失敗:", kt["rec_date"].isna().sum(), "/ 時刻パース失敗:", kt["rec_dt"].isna().sum())
print(kt.loc[kt["rec_date"].isna(), "visit_datetime"].head(10).tolist())

kt = kt.merge(ant[["Study_ID", "death_date"]], on="Study_ID", how="inner")
kt = kt[kt["rec_date"] <= kt["death_date"]].copy()
kt["source"] = np.where(kt["visit_type"] == NURSE_TYPE, "看護", "医師")
kt["text_n"] = kt["karte_text"].map(nfkc)
print("対象カルテ:", len(kt), "件 /", kt["Study_ID"].nunique(), "例")

In [ ]:
# 4. 死亡時刻（当院の死亡確認カルテの診察時刻）
vt_pat = "|".join(map(re.escape, DEATH_VISIT_TYPES))
dw_pat = "|".join(map(re.escape, DEATH_WORDS))
dd = kt[(kt["rec_date"] == kt["death_date"])
        & (kt["source"] == "医師")
        & kt["visit_type"].fillna("").str.contains(vt_pat)
        & kt["text_n"].str.contains(dw_pat)]
death_time = dd.groupby("Study_ID")["rec_dt"].min().rename("death_dt")
ant = ant.drop(columns="death_dt", errors="ignore").merge(death_time, on="Study_ID", how="left")
print("死亡時刻あり:", ant["death_dt"].notna().sum(), "/ なし（日単位判定）:", ant["death_dt"].isna().sum())

In [ ]:
# 5. ARISヒット抽出（1行=1カルテ記録）
rows = []
for r in kt.itertuples(index=False):
    ms = list(ARIS_RE.finditer(r.text_n))
    if not ms:
        continue
    t = r.text_n
    yotei = any("予定" in t[max(0, m.start() - YOTEI_WIN): m.end() + YOTEI_WIN] for m in ms)
    m0 = ms[0]
    rows.append(dict(
        Study_ID=r.Study_ID, record_no=r.record_no, source=r.source, visit_type=r.visit_type,
        rec_dt=r.rec_dt, rec_date=r.rec_date, n_match=len(ms),
        matches="|".join(sorted({m.group() for m in ms})), yotei=int(yotei),
        context=t[max(0, m0.start() - 60): m0.end() + 60].replace("\n", " "),
    ))
hits = pd.DataFrame(rows)
hits = hits.merge(ant[["Study_ID", "death_date", "death_dt"]], on="Study_ID", how="left")
hits["lead_days"]  = (hits["death_date"] - hits["rec_date"]).dt.days
hits["lead_hours"] = ((hits["death_dt"] - hits["rec_dt"]).dt.total_seconds() / 3600).round(1)
hits = hits.sort_values(["Study_ID", "rec_dt", "record_no"]).reset_index(drop=True)
hits["hit_order"] = hits.groupby("Study_ID").cumcount() + 1
print("ヒット記録:", len(hits), "/ ヒットあり患者:", hits["Study_ID"].nunique(), "/", len(ant))
print(hits["matches"].value_counts().head(20))

In [ ]:
# 6. 目視確認用CSV出力
review = hits.copy()
review.insert(0, "valid", "")   # 1=該当, 0=偽陽性。患者ごとに hit_order=1 から確認し、最初の1が出たら以降は空欄でよい
review.to_csv(OUT / "aris_hits_review.csv", index=False, encoding="utf-8-sig")

nohit = ant[~ant["Study_ID"].isin(hits["Study_ID"])]
nohit.to_csv(OUT / "aris_nohit.csv", index=False, encoding="utf-8-sig")
print("ヒットなし（据え置き）:", len(nohit))

In [ ]:
# 5b. 「今後」欄の見出し表記ゆれの確認（KONGO_HEAD の修正用）
doc = kt[(kt["source"] == "医師") & kt["visit_type"].fillna("").str.contains(TEIKI_TYPE)].copy()
print("医師・定期カルテ:", len(doc), "件 /", doc["Study_ID"].nunique(), "例")
HEAD_RE = re.compile(rf"[{MARK}]\s*(今後[^\s{MARK}:：、。]{{0,8}})")
print(doc["text_n"].str.findall(HEAD_RE).explode().value_counts().head(60).to_string())

In [ ]:
# 5c. 「今後」欄の抽出と、直前記録からの文言変化の検出
import difflib

SEC_RE = re.compile(rf"[{MARK}]\s*(?:{KONGO_HEAD})\s*[:：]?(.*?)(?=\s[{MARK}]|\Z)", re.S)

def get_sec(t):
    return " / ".join(m.group(1).strip() for m in SEC_RE.finditer(t)).strip()

def added_text(prev, cur):
    if not isinstance(prev, str):
        return cur
    sm = difflib.SequenceMatcher(None, prev, cur, autojunk=False)
    return " … ".join(cur[j1:j2] for op, i1, i2, j1, j2 in sm.get_opcodes() if op in ("insert", "replace"))

sec = doc.assign(kongo=doc["text_n"].map(get_sec))
sec = sec[sec["kongo"] != ""].sort_values(["Study_ID", "rec_dt", "record_no"]).copy()
sec["kongo_cmp"] = sec["kongo"].str.replace(r"\s+", "", regex=True)
g = sec.groupby("Study_ID")
sec["prev"]     = g["kongo"].shift()
sec["prev_cmp"] = g["kongo_cmp"].shift()
sec["change_type"] = np.where(sec["prev_cmp"].isna(), "初出",
                      np.where(sec["kongo_cmp"] != sec["prev_cmp"], "変更", ""))
print("「今後」欄あり:", len(sec), "件 /", sec["Study_ID"].nunique(), "例")
print(sec["change_type"].value_counts())

kongo = sec[sec["change_type"] != ""].copy()
kongo["added"] = [added_text(p, c) for p, c in zip(kongo["prev_cmp"], kongo["kongo_cmp"])]
kongo = kongo.merge(ant[["Study_ID", "death_dt"]], on="Study_ID", how="left")
kongo["lead_days"]  = (kongo["death_date"] - kongo["rec_date"]).dt.days
kongo["lead_hours"] = ((kongo["death_dt"] - kongo["rec_dt"]).dt.total_seconds() / 3600).round(1)
kongo["hit_order"]  = kongo.groupby("Study_ID").cumcount() + 1

In [ ]:
# 6b. 「今後」欄の目視確認用CSV出力
cols = ["Study_ID", "record_no", "source", "visit_type", "rec_dt", "rec_date", "death_date", "death_dt",
        "lead_days", "lead_hours", "hit_order", "change_type", "added", "prev", "kongo"]
kr = kongo[cols].copy()
kr.insert(0, "valid", "")   # 1=終末期の認識を示す変化, 0=該当しない。患者ごとに hit_order=1 から確認
kr.to_csv(OUT / "kongo_review.csv", index=False, encoding="utf-8-sig")
print("目視候補:", len(kr), "件 /", kr["Study_ID"].nunique(), "例")

---
## Phase 2：aris_hits_review.csv の valid 列を記入後に実行

In [ ]:
# 7. first_date確定（ARIS＋「今後」欄）と閾値フラグ
def load_review(fn, kind, evid_col):
    p = OUT / fn
    if not p.exists():
        print(f"{fn} なし（スキップ）")
        return None
    d = pd.read_csv(p, encoding="utf-8-sig", parse_dates=["rec_dt", "rec_date", "death_date", "death_dt"])
    d["valid"] = pd.to_numeric(d["valid"], errors="coerce")
    d["kind"] = kind
    d["evidence"] = d[evid_col]
    if "yotei" not in d:
        d["yotei"] = np.nan
    return d[["Study_ID", "record_no", "kind", "source", "rec_dt", "rec_date",
              "lead_days", "lead_hours", "yotei", "evidence", "valid"]]

rev = pd.concat([x for x in [load_review("aris_hits_review.csv", "ARIS", "context"),
                             load_review("kongo_review.csv", "今後", "added")] if x is not None],
                ignore_index=True)
print("記入済み:", rev["valid"].notna().sum(), "/", len(rev))

first = (rev[rev["valid"] == 1].sort_values(["Study_ID", "rec_dt"])
            .groupby("Study_ID").head(1)
            .rename(columns={"rec_dt": "first_dt", "rec_date": "first_date"}))

# 確定した first より前に未確認（空欄）の記録が残っていないか（両ファイル横断）
chk = rev.merge(first[["Study_ID", "first_dt"]], on="Study_ID")
unchecked = chk[(chk["rec_dt"] < chk["first_dt"]) & chk["valid"].isna()]
print("first より前の未確認記録:", len(unchecked))

vv = rev.groupby("Study_ID")["valid"]
all_false = vv.apply(lambda s: s.notna().all() and (s == 0).all())
print("全ヒット偽陽性:", int(all_false.sum()))
print("first_date 確定:", len(first), "/", len(ant))

res = ant.merge(first[["Study_ID", "first_dt", "first_date", "kind", "source", "yotei",
                       "lead_days", "lead_hours", "evidence"]], on="Study_ID", how="left")
has_time = res["death_dt"].notna() & res["first_dt"].notna()
res["judge_unit"] = np.where(res["first_date"].isna(), "", np.where(has_time, "hour", "day"))

for k, h in THRESH_H.items():
    d = h // 24
    ge = np.where(has_time, res["lead_hours"] >= h, res["lead_days"] >= d).astype(float)
    ge[res["first_date"].isna().to_numpy()] = np.nan
    res[f"ge_{k}"] = ge
    res[f"border_{k}"] = ((~has_time) & (res["lead_days"] == d)).astype(int)

res.to_csv(OUT / "aris_firstdate_result.csv", index=False, encoding="utf-8-sig")

print(res["kind"].value_counts(dropna=False))
print(res["lead_days"].describe())
summ = pd.DataFrame({k: [int(np.nansum(res[f"ge_{k}"])), int(res[f"ge_{k}"].notna().sum()),
                         int(res[f"border_{k}"].sum())] for k in THRESH_H},
                    index=["記載あり(≥閾値)", "first_dateあり", "境界フラグ"])
print(summ)